In [14]:
import requests
from datetime import datetime
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from dotenv import load_dotenv, find_dotenv 
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

from download_cards import download_model_cards
from utils import *

from retriever import Retriever
from generator import Generator

_ = load_dotenv(find_dotenv())

# Indexing 

In [ ]:
repo_url = "https://github.com/evidentlyai/evidently"
repo_name = repo_url.rstrip('/').split('/')[-1]
extract_dir = f"./{repo_name}"

download_repo = True
if download_repo:
    download_and_extract_repo(repo_url, extract_dir)

py_files = get_py_files(extract_dir)

In [ ]:
# Step 1.1: load documents
#loader = DirectoryLoader('model_cards/', glob="**/*.md", loader_cls=TextLoader)
loader = DirectoryLoader('evidently/', glob="**/*.py", loader_cls=TextLoader)
documents = loader.load()
# Step 1.2: split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

In [5]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)
# Step 1.3: encode chunks into vectors and store in a vector database
vectordb = FAISS.from_documents(chunks, embeddings)


# Retrieval

In [10]:
# Step 2: Retrieval: retrieve the Top k chunks most relevant to the question based on semantic similarity.
retriever = vectordb.as_retriever()

# Generation

In [11]:
generator = Generator(model="gpt-4o")
template = generator.format_prompt_codegen(
    system_prompt_path="prompts/system_prompt_codegen.txt", 
    user_prompt_path="prompts/user_prompt_codegen.txt")
template

"You are a helpful assistant that generates Python code based on the provided context.\nConsider the following definitions:\n    1) Data validation is an operation in the AI system lifecycle that validates the quality of training data to identify bias issues that could impact model performance across groups of sensitive features.\n    2) Data preprocessing is an operation in the AI system lifecycle that applies data cleaning procedure, data augmentation, type conversion. This operation can analyze data distribution, evaluate bias or discrimination issues on data and transform data in order to mitigate potential risks deriving from low data quality.\n    3) Fairness means ensuring equity in the decision-making process of a machine learning algorithm across individuals and groups. Group fairness split a population into groups defined by protected attributes (e.g. gender, race) and seeks for some measure to be as equal as possible across groups. Some fairness metrics for measuring group f

## Prompt Engineering
- TODO: DSPy prompt optimization (query parsing)

In [12]:
prompt = ChatPromptTemplate.from_template(template)
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are a helpful assistant that generates Python code based on the provided context.\nConsider the following definitions:\n    1) Data validation is an operation in the AI system lifecycle that validates the quality of training data to identify bias issues that could impact model performance across groups of sensitive features.\n    2) Data preprocessing is an operation in the AI system lifecycle that applies data cleaning procedure, data augmentation, type conversion. This operation can analyze data distribution, evaluate bias or discrimination issues on data and transform data in order to mitigate potential risks deriving from low data quality.\n    3) Fairness means ensuring equity in the decision-making process of a machine learning algorithm across individuals

In [16]:
# Step 3: Generation: input the original question and the retrieved chunks together into LLM to generate the final answer.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.5)

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()} 
    | prompt 
    | llm
    | StrOutputParser() 
)

query = "Please generate a new python script that detects data drift for tabular data using your context?"
result = rag_chain.invoke(query)
current_time = datetime.now()
with open(f"results/generated_{current_time}.py", "w") as file:
    file.write(result)

# RAG Evaluation

In [ ]:
# Import necessary libraries
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Instantiate the models
generator_llm = ChatOpenAI(model="gpt-4o-mini")
critic_llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

# Create the TestsetGenerator
generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    embeddings
)

# Call the generator
testset = generator.generate_with_langchain_docs(
data_transformed, 
test_size=20, 
distributions={ 
simple: 0.5, 
reasoning: 0.25, 
multi_context: 0.25}
)

# Agentic RAG